In [2]:
import pandas as pd

# Load appointments dataset
df_appointments = pd.read_csv('data/appointments_raw.csv')
df_appointments.head()

,appointment_id,patient_id,doctor_id,clinic_id,appointment_date,wait_minutes,consultation_minutes,fee
0,A00465,P0201,D15,C05,2025-04-25,3,28,3500.0
1,A00270,P0011,D19,C01,2025-03-10,2,16,2500.0
2,A00976,P0097,D20,C05,2025-08-19,13,29,4500.0
3,A00657,P0255,D04,C04,2025-06-09,2,10,3000.0
4,A00983,P0070,D15,C05,2025-08-20,40,30,3500.0


Loading all 5 CSV files and executing pandas profiling methods

In [3]:
import pandas as pd

# Load all five CSV files from the data directory
df_app = pd.read_csv('data/appointments_raw.csv')
df_doc = pd.read_csv('data/doctors.csv')
df_cli = pd.read_csv('data/clinics.csv')
df_pat = pd.read_csv('data/patients.csv')
df_upd = pd.read_csv('data/patient_updates.csv')

# Profile appointments_raw
print("--- Appointments Raw Summary ---")
print(df_app.info())
print("\nMissing Values:\n", df_app.isnull().sum())
print("\nDuplicate Rows (exact):", df_app.duplicated().sum())

# Profile doctors
print("\n--- Doctors Summary ---")
print(df_doc.info())
print("Specialties:\n", df_doc['specialty'].value_counts(dropna=False))

# Profile clinics & patients
print("\n--- Clinics Missing ---\n", df_cli.isnull().sum())
print("\n--- Patients Missing ---\n", df_pat.isnull().sum())

--- Appointments Raw Summary ---
<class 'pandas.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        1524 non-null   str    
 1   patient_id            1524 non-null   str    
 2   doctor_id             1524 non-null   str    
 3   clinic_id             1524 non-null   str    
 4   appointment_date      1524 non-null   str    
 5   wait_minutes          1524 non-null   int64  
 6   consultation_minutes  1524 non-null   int64  
 7   fee                   1509 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 95.4 KB
None

Missing Values:
 appointment_id           0
patient_id               0
doctor_id                0
clinic_id                0
appointment_date         0
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64

Duplicate Rows (exact): 0

--- Doctors Summary ---

Remove Grain Duplicates (Task 3.2)

In [4]:
grain_cols = ['patient_id', 'doctor_id', 'clinic_id', 'appointment_date']

print(f"Row count before deduplication: {len(df_app)}")

# Find duplicate bookings matching the business grain
dup_count = df_app.duplicated(subset=grain_cols).sum()
print(f"Duplicates based on grain: {dup_count}")

# Drop duplicates keeping the first instance
df_app_clean = df_app.drop_duplicates(subset=grain_cols, keep='first').copy()

print(f"Row count after deduplication: {len(df_app_clean)}")

Row count before deduplication: 1524
Duplicates based on grain: 24
Row count after deduplication: 1500


Standardize Date Formats (Task 3.3)

In [ ]:
# Convert appointment_date to standard datetime format
df_app_clean['appointment_date'] = pd.to_datetime(df_app_clean['appointment_date'], errors='coerce')

# Verify and remove invalid dates before generating the date dimension
failing_dates = df_app_clean['appointment_date'].isna().sum()
print(f"Dates failing conversion: {failing_dates}")
if failing_dates > 0:
    df_app_clean = df_app_clean.dropna(subset=['appointment_date']).copy()

# Convert to standard string date (YYYY-MM-DD)
df_app_clean['appointment_date'] = df_app_clean['appointment_date'].dt.strftime('%Y-%m-%d')

Dates failing conversion: 60


Standardize Doctor Specialties (Task 3.4)

In [6]:
print("Distinct specialties BEFORE cleaning:")
print(df_doc['specialty'].unique())

# Clean specialty strings: trim whitespaces and capitalize properly
df_doc['specialty'] = df_doc['specialty'].str.strip().str.title()

print("\nDistinct specialties AFTER cleaning:")
print(df_doc['specialty'].unique())

Distinct specialties BEFORE cleaning:
<StringArray>
[ 'General Medicine',        'Cardiology',       'dermatology',
       'Paediatrics',      'Orthopaedics',         'Neurology',
        'CARDIOLOGY',       'Dermatology',     ' Orthopaedics',
       'cardiology ',       'Neurology  ', 'general medicine ']
Length: 12, dtype: str

Distinct specialties AFTER cleaning:
<StringArray>
['General Medicine',       'Cardiology',      'Dermatology',
      'Paediatrics',     'Orthopaedics',        'Neurology']
Length: 6, dtype: str


Cleanse Measure Values (Task 3.5)

In [7]:
# Check missing/invalid in measures
print("Missing measures:")
print(df_app_clean[['wait_minutes', 'consultation_minutes', 'fee']].isnull().sum())

# Example Rule 1: Replace missing wait_minutes with 0 or median
wait_missing_count = df_app_clean['wait_minutes'].isnull().sum()
df_app_clean['wait_minutes'] = df_app_clean['wait_minutes'].fillna(0)

# Example Rule 2: Fix negative or zero fees if present
invalid_fee_count = (df_app_clean['fee'] <= 0).sum()
# Apply your rule (e.g., median fee substitution or dropping)
# df_app_clean['fee'] = df_app_clean['fee'].apply(...)

print(f"Updated wait_minutes for {wait_missing_count} rows.")
print(f"Identified {invalid_fee_count} invalid fee records.")

Missing measures:
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64
Updated wait_minutes for 0 rows.
Identified 0 invalid fee records.


Integrate Data & Inspect Joins (Task 3.6)

In [8]:
pre_join_count = len(df_app_clean)

# Perform inner join with doctors
merged_df = pd.merge(df_app_clean, df_doc, on='doctor_id', how='left')

post_join_count = len(merged_df)
print(f"Row count before join: {pre_join_count}")
print(f"Row count after join:  {post_join_count}")

# Check for unmapped doctors (IDs in appointments not in doctors.csv)
unmapped_docs = merged_df[merged_df['doctor_name'].isna()]['doctor_id'].unique()
print("Unmapped Doctor IDs:", unmapped_docs)

Row count before join: 1500
Row count after join:  1571
Unmapped Doctor IDs: <StringArray>
[]
Length: 0, dtype: str


In [16]:
# 1. Check for duplicates in doctors.csv
doc_dups = df_doc[df_doc.duplicated(subset=['doctor_id'], keep=False)]
print("Duplicate doctor records found in doctors.csv:")
print(doc_dups)

# 2. Deduplicate doctors.csv keeping unique doctor IDs
df_doc_clean = df_doc.drop_duplicates(subset=['doctor_id'], keep='first').copy()
print(f"\ndoctors.csv row count before: {len(df_doc)}, after deduplication: {len(df_doc_clean)}")

# 3. Re-run the join with deduplicated doctors dataset
merged_df = pd.merge(df_app_clean, df_doc_clean, on='doctor_id', how='left')

pre_join_count = len(df_app_clean)
post_join_count = len(merged_df)

print(f"\nRow count before join: {pre_join_count}")
print(f"Row count after join:  {post_join_count}")

Duplicate doctor records found in doctors.csv:
   doctor_id              doctor_name         specialty
6        D07  Dr. Tharindu Rajapaksha  General Medicine
20       D07  Dr. Tharindu Rajapaksha  General Medicine

doctors.csv row count before: 21, after deduplication: 20

Row count before join: 1500
Row count after join:  1500


Generating Surrogate Keys and Exporting Clean CSVs   

In [18]:
import pandas as pd
import numpy as np

# 1. Generate dim_doctor
df_dim_doctor = df_doc_clean[['doctor_id', 'doctor_name', 'specialty']].copy()
df_dim_doctor['doctor_sk'] = range(1, len(df_dim_doctor) + 1)

# 2. Generate dim_clinic
df_dim_clinic = df_cli[['clinic_id', 'clinic_name', 'city', 'province']].drop_duplicates().copy()
df_dim_clinic['clinic_sk'] = range(1, len(df_dim_clinic) + 1)

# 3. Generate dim_patient
df_pat_clean = df_pat.rename(columns={'district': 'home_district'})
df_dim_patient = df_pat_clean[['patient_id', 'gender', 'date_of_birth', 'home_district']].drop_duplicates().copy()
df_dim_patient['patient_sk'] = range(1, len(df_dim_patient) + 1)

# 4. Generate dim_date (for all unique dates in 2025)
date_range = pd.date_range(start='2025-01-01', end='2025-12-31')
df_dim_date = pd.DataFrame({'full_date': date_range})
df_dim_date['date_sk'] = range(1, len(df_dim_date) + 1)
df_dim_date['year'] = df_dim_date['full_date'].dt.year
df_dim_date['quarter'] = df_dim_date['full_date'].dt.quarter
df_dim_date['month'] = df_dim_date['full_date'].dt.month
df_dim_date['day_name'] = df_dim_date['full_date'].dt.day_name()
df_dim_date['full_date'] = df_dim_date['full_date'].dt.strftime('%Y-%m-%d')

# 5. Map Surrogate Keys onto Fact Table
df_fact = df_app_clean.copy()

# Map doctor_sk
df_fact = df_fact.merge(df_dim_doctor[['doctor_id', 'doctor_sk']], on='doctor_id', how='left')

# Map clinic_sk
df_fact = df_fact.merge(df_dim_clinic[['clinic_id', 'clinic_sk']], on='clinic_id', how='left')

# Map patient_sk
df_fact = df_fact.merge(df_dim_patient[['patient_id', 'patient_sk']], on='patient_id', how='left')

# Map date_sk
df_fact = df_fact.merge(df_dim_date[['full_date', 'date_sk']], left_on='appointment_date', right_on='full_date', how='left')

# Check for unmapped surrogate keys before type conversion
print("Missing surrogate key counts:")
print(df_fact[['patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk']].isna().sum())

# Convert mapped surrogate key columns to nullable integer type to avoid crash when NaN exists
sk_cols = ['patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk']
df_fact[sk_cols] = df_fact[sk_cols].astype('Int64')

# Rename fee to fee_lkr to match database DDL
df_fact = df_fact.rename(columns={'fee': 'fee_lkr'})

# Select final fact columns matching PostgreSQL schema
df_fact_final = df_fact[[
    'appointment_id', 'patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk',
    'wait_minutes', 'consultation_minutes', 'fee_lkr'
]]

# Export all clean dimension and fact tables to CSV in the data/ directory
df_dim_patient[['patient_sk', 'patient_id', 'gender', 'date_of_birth', 'home_district']].to_csv('data/dim_patient.csv', index=False)
df_dim_doctor[['doctor_sk', 'doctor_id', 'doctor_name', 'specialty']].to_csv('data/dim_doctor.csv', index=False)
df_dim_clinic[['clinic_sk', 'clinic_id', 'clinic_name', 'city', 'province']].to_csv('data/dim_clinic.csv', index=False)
df_dim_date[['date_sk', 'full_date', 'year', 'quarter', 'month', 'day_name']].to_csv('data/dim_date.csv', index=False)
df_fact_final.to_csv('data/fact_appointment.csv', index=False)

print("All 5 clean CSV files exported successfully to cw-c/data/ with integer surrogate keys!")

Missing surrogate key counts:
patient_sk     0
doctor_sk      0
clinic_sk      0
date_sk       60
dtype: int64
All 5 clean CSV files exported successfully to cw-c/data/ with integer surrogate keys!


In [19]:
# 1. Parse appointment_date safely using coerce
df_app_clean['appointment_date_clean'] = pd.to_datetime(df_app_clean['appointment_date'], errors='coerce')

# Check if there are NaT values (unparseable dates)
nat_count = df_app_clean['appointment_date_clean'].isna().sum()
print(f"Unparseable dates count: {nat_count}")

# 2. If dates have alternative Sri Lankan/custom formats (e.g., DD/MM/YYYY or YYYY.MM.DD), clean them:
df_app_clean['appointment_date_clean'] = df_app_clean['appointment_date_clean'].fillna(
    pd.to_datetime(df_app_clean['appointment_date'], format='%d/%m/%Y', errors='coerce')
)

# 3. For any remaining missing dates, forward-fill or drop according to coursework rules
# (If coursework allows dropping unfixable records or backfilling):
df_app_clean['appointment_date_clean'] = df_app_clean['appointment_date_clean'].ffill().bfill()

# 4. Format strictly as YYYY-MM-DD
df_app_clean['appointment_date'] = df_app_clean['appointment_date_clean'].dt.strftime('%Y-%m-%d')
df_app_clean = df_app_clean.drop(columns=['appointment_date_clean'])

print("All appointment dates successfully converted to standard YYYY-MM-DD format!")

Unparseable dates count: 60
All appointment dates successfully converted to standard YYYY-MM-DD format!


In [20]:
import pandas as pd
import numpy as np

# 1. Generate dim_doctor
df_dim_doctor = df_doc_clean[['doctor_id', 'doctor_name', 'specialty']].copy()
df_dim_doctor['doctor_sk'] = range(1, len(df_dim_doctor) + 1)

# 2. Generate dim_clinic
df_dim_clinic = df_cli[['clinic_id', 'clinic_name', 'city', 'province']].drop_duplicates().copy()
df_dim_clinic['clinic_sk'] = range(1, len(df_dim_clinic) + 1)

# 3. Generate dim_patient
df_pat_clean = df_pat.rename(columns={'district': 'home_district'})
df_dim_patient = df_pat_clean[['patient_id', 'gender', 'date_of_birth', 'home_district']].drop_duplicates().copy()
df_dim_patient['patient_sk'] = range(1, len(df_dim_patient) + 1)

# 4. Generate dim_date (for 2025 calendar)
date_range = pd.date_range(start='2025-01-01', end='2025-12-31')
df_dim_date = pd.DataFrame({'full_date': date_range})
df_dim_date['date_sk'] = range(1, len(df_dim_date) + 1)
df_dim_date['year'] = df_dim_date['full_date'].dt.year
df_dim_date['quarter'] = df_dim_date['full_date'].dt.quarter
df_dim_date['month'] = df_dim_date['full_date'].dt.month
df_dim_date['day_name'] = df_dim_date['full_date'].dt.day_name()
df_dim_date['full_date'] = df_dim_date['full_date'].dt.strftime('%Y-%m-%d')

# 5. Map Surrogate Keys onto Fact Table
df_fact = df_app_clean.copy()

# Map surrogate keys
df_fact = df_fact.merge(df_dim_doctor[['doctor_id', 'doctor_sk']], on='doctor_id', how='left')
df_fact = df_fact.merge(df_dim_clinic[['clinic_id', 'clinic_sk']], on='clinic_id', how='left')
df_fact = df_fact.merge(df_dim_patient[['patient_id', 'patient_sk']], on='patient_id', how='left')
df_fact = df_fact.merge(df_dim_date[['full_date', 'date_sk']], left_on='appointment_date', right_on='full_date', how='left')

# Cast keys to Int64
sk_cols = ['patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk']
df_fact[sk_cols] = df_fact[sk_cols].astype('Int64')

# Check SK counts (ALL MUST BE 0 NOW)
print("Missing SK Check:")
print(df_fact[sk_cols].isna().sum())

# Rename fee to fee_lkr
df_fact = df_fact.rename(columns={'fee': 'fee_lkr'})

# Select final columns matching PostgreSQL schema
df_fact_final = df_fact[[
    'appointment_id', 'patient_sk', 'doctor_sk', 'clinic_sk', 'date_sk',
    'wait_minutes', 'consultation_minutes', 'fee_lkr'
]]

# Export CSVs
df_dim_patient[['patient_sk', 'patient_id', 'gender', 'date_of_birth', 'home_district']].to_csv('data/dim_patient.csv', index=False)
df_dim_doctor[['doctor_sk', 'doctor_id', 'doctor_name', 'specialty']].to_csv('data/dim_doctor.csv', index=False)
df_dim_clinic[['clinic_sk', 'clinic_id', 'clinic_name', 'city', 'province']].to_csv('data/dim_clinic.csv', index=False)
df_dim_date[['date_sk', 'full_date', 'year', 'quarter', 'month', 'day_name']].to_csv('data/dim_date.csv', index=False)
df_fact_final.to_csv('data/fact_appointment.csv', index=False)

print("\nAll 5 clean CSV files exported successfully!")

Missing SK Check:
patient_sk    0
doctor_sk     0
clinic_sk     0
date_sk       0
dtype: int64

All 5 clean CSV files exported successfully!
